# Recall Analysis

`doc_id_list` / `is_gold_list`에서 중복 `doc_id`를 제거한 gold hit 수를 세고, `num_hops` 기준 recall을 계산합니다.

- `macro recall`: 샘플별 recall의 평균
- `micro recall`: 전체 gold hit 합 / 전체 hop 합

In [2]:
from pathlib import Path
import json
import pandas as pd


def load_records(input_path):
    input_path = Path(input_path)
    with input_path.open('r', encoding='utf-8') as f:
        return json.load(f)


def normalize_doc_id(doc_id):
    return str(doc_id).strip()


def collect_gold_doc_ids(sample):
    gold_doc_ids = set()

    step_trace = sample.get('step_trace') or []
    for step in step_trace:
        info = step.get('step_retrieval_info') or {}
        doc_ids = info.get('doc_id_list') or []
        is_gold_list = info.get('is_gold_list') or []
        for doc_id, is_gold in zip(doc_ids, is_gold_list):
            if is_gold == 1:
                gold_doc_ids.add(normalize_doc_id(doc_id))

    if gold_doc_ids:
        return gold_doc_ids

    for key, value in sample.items():
        if not key.startswith('retrieval_'):
            continue
        if not isinstance(value, dict):
            continue
        doc_ids = value.get('doc_id_list') or []
        is_gold_list = value.get('is_gold_list') or []
        for doc_id, is_gold in zip(doc_ids, is_gold_list):
            if is_gold == 1:
                gold_doc_ids.add(normalize_doc_id(doc_id))

    return gold_doc_ids


def build_recall_dataframe(records):
    rows = []
    for sample in records:
        num_hops = sample.get('num_hops')
        if not isinstance(num_hops, int) or num_hops <= 0:
            continue

        gold_doc_ids = collect_gold_doc_ids(sample)
        unique_gold_hits = len(gold_doc_ids)
        capped_gold_hits = min(unique_gold_hits, num_hops)
        sample_recall = capped_gold_hits / num_hops

        rows.append({
            'index': sample.get('index'),
            'uid': sample.get('uid'),
            'num_hops': num_hops,
            'unique_gold_hits': unique_gold_hits,
            'capped_gold_hits': capped_gold_hits,
            'recall': sample_recall,
            'gold_doc_ids': sorted(gold_doc_ids),
        })

    return pd.DataFrame(rows)


def build_recall_summary(df):
    if len(df) == 0:
        return pd.DataFrame([{
            'num_samples': 0,
            'recall': 0.0,
            'total_num_hops': 0,
        }])

    return pd.DataFrame([{
        '':"musique_triplet",
        'num_samples': len(df),
        'recall': (df['capped_gold_hits'].sum() / df['num_hops'].sum()).round(2),
        'total_num_hops': int(df['num_hops'].sum()),
    }])


In [77]:
input_path = '/home/hyeseojeon/data/graph/results/veri/0410/2wiki_graph_reasoning_path_searchr1_graph_open-book_stepwise_129249_500.json'

records = load_records(input_path)
df = build_recall_dataframe(records)
summary = build_recall_summary(df)

summary

,,num_samples,recall,total_num_hops
0,2wiki_triplet,500,0.32,1186


In [79]:
input_path = '/home/hyeseojeon/data/graph/results/vanilla/0407(open-book)/2wiki_vanilla_searchr1_128615_500.json'

records = load_records(input_path)
df = build_recall_dataframe(records)
summary = build_recall_summary(df)

summary

,,num_samples,recall,total_num_hops
0,2wiki_vanilla,500,0.33,1186


In [81]:
input_path = '/home/hyeseojeon/data/graph/results/veri/0410/hotpotqa_graph_reasoning_path_searchr1_graph_open-book_stepwise_129250_500.json'

records = load_records(input_path)
df = build_recall_dataframe(records)
summary = build_recall_summary(df)

summary

,,num_samples,recall,total_num_hops
0,hotpotqa_triplet,500,0.49,1000


In [83]:
input_path = '/home/hyeseojeon/data/graph/results/vanilla/0407(open-book)/hotpotqa_vanilla_searchr1_128616_500.json'

records = load_records(input_path)
df = build_recall_dataframe(records)
summary = build_recall_summary(df)

summary

,,num_samples,recall,total_num_hops
0,hotpotqa_vanilla,500,0.52,1000


In [86]:
input_path = '/home/hyeseojeon/data/graph/results/vanilla/0407(open-book)/musique_vanilla_searchr1_128617_1000.json'

records = load_records(input_path)
df = build_recall_dataframe(records)
summary = build_recall_summary(df)

summary

,,num_samples,recall,total_num_hops
0,musique_vanilla,1000,0.22,2900


In [6]:
input_path = '/home/hyeseojeon/data/graph/results/veri/0410/musique_graph_reasoning_path_searchr1_graph_open-book_stepwise_129248_1000.json'

records = load_records(input_path)
df = build_recall_dataframe(records)
summary = build_recall_summary(df)

summary

,,num_samples,recall,total_num_hops
0,musique_triplet,1000,0.21,2900
